In [14]:
# Import

from anthropic import Anthropic
from dotenv import load_dotenv

# Env var
load_dotenv()


True

## Notebook Goal
In this notebook, you will compare two ways to stream Claude responses and learn how to capture the final message for storage.

Run the setup cells first so your client is initialized before testing streaming examples.

In [15]:
# Define modle and client
model = "claude-haiku-4-5-20251001"

client = Anthropic()

## Setup Notes
This model/client setup is reused for each example. Keeping it in one place makes it easy to switch models later without rewriting downstream cells.

In [16]:
## Define helper fucntions 
def add_user_message(messages, text):
    return messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    return messages.append({"role": "assistant", "content": text})

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    # Only include system when present.
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Raw Event Streaming
This version uses `client.messages.create(..., stream=True)` and prints every stream event.

Use this when you want full visibility into event types like `ContentBlockDelta`, `MessageDelta`, and `MessageStop`.

In [17]:
# User message 
messages = []

add_user_message(
    messages,
    "Write a 1 sentence description of a fake database"
    )

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)



RawMessageStartEvent(message=Message(id='msg_011CeiGTdVFEAB4K8pQst7u1', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' F', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='akeDB\n\nA lightweight in-memory database engine that gener

## Simplified Text Streaming
The SDK helper `client.messages.stream(...)` exposes `text_stream`, which gives you only generated text chunks.

This is the easiest way to render tokens in a UI without manual event parsing.

In [18]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

# FakeDB

FakeDB is an in-memory database system that generates and stores randomly procedurally-generated data structures to simulate real database operations without requiring actual persistent storage or network connectivity.

## Capture the Final Message
Streaming improves user experience, but applications often also need the complete assistant message for logs, analytics, or database storage.

Use `stream.get_final_message()` after the stream loop to retrieve the full response object.

In [19]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass

final_message = stream.get_final_message()
final_message.content[0].text

'# FakeDB\n\nA lightweight, in-memory database system that generates and stores randomly-seeded mock data for testing applications without requiring external dependencies or actual data infrastructure.'

## Optional Next Step
Assign the result of `stream.get_final_message()` to a variable and extract the text you want to persist, such as `final_message.content[0].text`.